In [ ]:
import os
import warnings

if 'Modeling' in os.path.abspath("").split('/'):
    os.chdir('..')
if 'Notebooks' in os.path.abspath("").split('/'):
    os.chdir('..')

project_root = os.path.abspath("")

warnings.filterwarnings('ignore')

In [ ]:
# import numpy as np
# loaded = np.load('./Generated/Spectrums/exec_morlets.npz')

In [ ]:
# from Scripts.Data_Loader import EIRDataset

# EIR_Dataset = EIRDataset('./Generated/Data_Train/', task_type='geometric', n_jobs=72) # task type can be `geometric` or `random` or `all`

In [ ]:
# results_arr = []
# i = 0
# while f'power_{i}' in loaded:
#     power = loaded[f'power_{i}']
#     phase = loaded[f'phase_{i}']
#     s_id = int(loaded[f'subject_id_{i}'])
#     t_id = int(loaded[f'trial_id_{i}'])
#     gender = str(loaded[f'gender_{i}'])
#     handiness = str(loaded[f'handiness_{i}'])
#     age = int(loaded[f'age_{i}'])
#     label = int(loaded[f'label_{i}'])
#     img = loaded[f'img_{i}']
#     task_type = str(loaded[f'task_type_{i}'])
    
#     results_arr.append([power, phase, s_id, t_id, gender, handiness, age, label, img, task_type])
#     i += 1

# power, phase, s_id, t_id, gender, handiness, age, label, img, task_type = results_arr[0]

In [ ]:
from torch.utils.data import Dataset as TorchDataset
import torch
import pandas as pd
import numpy as np
import json
from tqdm import tqdm
import mne
from multiprocessing import Pool, cpu_count
from enum import Enum
import re

In [ ]:
class MorletDataset(TorchDataset):
    def __init__(self, dataset_path: str, morlet_data_path: str, task_type : str = 'all', n_jobs : int = 8, eeg_resampling_freq: int = None):
        # Raw data
        self.eeg_data = torch.Tensor()
        self.eye_data = torch.Tensor()
        self.metadata = pd.DataFrame(columns=["index", "subject_id", "trial_id", "task_type"])

        self.labels = torch.Tensor()
        self.imgs = torch.Tensor()
        self.__load_dataset_data(dataset_path, task_type, n_jobs, eeg_resampling_freq)

        # Morlet data
        self.power = torch.Tensor()
        self.phase = torch.Tensor()
        self.__load_morlet_data(morlet_data_path)

        
    
    def __load_dataset_data(self, dataset_path: str, task_type:str, n_jobs: int, eeg_resampling_freq: int) -> None:
        all_tasks = []
        index = 0
        labels = []
        imgs = []

        exp_paths = self.__collect_valid_paths(dataset_path)

        for exp_path in tqdm(exp_paths, desc="Processing exp_paths"):
            subject_id = int(os.path.basename(os.path.dirname(exp_path)).replace("S_", ""))
            trial_id = int(os.path.basename(exp_path).replace("Trial_", ""))
            with open(os.path.join(exp_path, "labels.json"), "r") as f:
                labels_data = json.load(f)["blocks"]

            for block in labels_data:
                task_type = block["type"]
                if task_type == task_type or task_type == 'all':
                    exec_idx = block["Exec_Block_Index"]
                    pattern_id = block.get("pattern_id", -1)

                    eeg_path = os.path.join(exp_path, f"exec_EEG_{exec_idx}.fif")
                    eye_path = os.path.join(exp_path, f"exec_EOG_{exec_idx}.fif")

                    if not (os.path.exists(eeg_path) and os.path.exists(eye_path)):
                        print(f"Skipping: EEG or EOG file missing for exec_idx={exec_idx} in {exp_path}")
                        continue

                    self.metadata.loc[len(self.metadata)] = {
                        "index": index,
                        "subject_id": subject_id,
                        "trial_id": trial_id,
                        "task_type": task_type
                    }
                    labels.append(pattern_id)
                    imgs.append(np.array(block["img"], dtype=np.float32))
                    all_tasks.append((eeg_path, eeg_resampling_freq, eye_path, index))
                    index += 1

        self.labels = torch.tensor(labels, dtype=torch.long)
        self.imgs = torch.from_numpy(np.stack(imgs))

        # Параллельная загрузка данных
        with Pool(n_jobs) as pool:
            results = list(tqdm(pool.imap(self.load_fif_wrapper, all_tasks), total=len(all_tasks), desc="Loading .fif files"))

        self.eeg_data = torch.zeros(len(results), *(results[0].get_data().shape), dtype=torch.float32)
        self.eye_data = torch.zeros(len(results), *(results[1].get_data().shape), dtype=torch.float32)
        # Распакуем результаты в torch.Tensor()
        for i, (eeg, eye) in enumerate(results):
            self.eeg_data[i] = torch.tensor(eeg.get_data(), dtype=torch.float32)
            self.eye_data[i] = torch.tensor(eye.get_data(), dtype=torch.float32)

    def __collect_valid_paths(self, root_dir: str):
        valid_dirs = []
        
        # Регулярка для S_директорий: S_ и только цифры после
        s_dir_pattern = re.compile(r"^S_\d+$")
        # Регулярка для Trial_директорий: Trial_ и только цифры
        trial_dir_pattern = re.compile(r"^Trial_\d+$")
        
        for s_dir in os.listdir(root_dir):
            s_path = os.path.join(root_dir, s_dir)
            if os.path.isdir(s_path) and s_dir_pattern.match(s_dir):
                for trial_dir in os.listdir(s_path):
                    trial_path = os.path.join(s_path, trial_dir)
                    if os.path.isdir(trial_path) and trial_dir_pattern.match(trial_dir):
                        # Проверяем, есть ли хотя бы один файл
                        if any(os.path.isfile(os.path.join(trial_path, f)) for f in os.listdir(trial_path)):
                            valid_dirs.append(trial_path)
        return valid_dirs

    def load_fif_wrapper(self, args) -> tuple[mne.io.Raw, mne.io.Raw]:
        eeg_path, eeg_resampling_freq, eye_path, index,  = args
        print("Start read raw")

        eeg = mne.io.read_raw_fif(eeg_path, preload=True, verbose='ERROR')
        eye = mne.io.read_raw_fif(eye_path, preload=True, verbose='ERROR')

        print("Raw read")

        if eeg_resampling_freq is not None:
            print("Resampling")
            eeg.resample(eeg_resampling_freq)
            print("Resampling finished")

        return eeg, eye

    def __load_morlet_data(self, morlet_data_path: str, max_len = 309) -> None:
        loaded = np.load(morlet_data_path)

        results_list = []
        i = 0
        print("Loading morlet data started")
        while f'power_{i}' in loaded:
            power = loaded[f'power_{i}'] 
            phase = loaded[f'phase_{i}']
            
            results_list.append([power, phase])
            i += 1
        
        print("Morlet data loaded")

        power_shape = tuple(len(results_list), *(results_list[0][0].shape[:-1]), max_len)
        self.power = torch.zeros(*power_shape, dtype=torch.float32)

        for i, sample in enumerate(results_list):
            if i == len(self.metadata):
                break

            self.power[i] = torch.from_numpy(sample[0][:, :, :max_len])
            self.phase[i] = torch.from_numpy(sample[1][:, :, :max_len])
        
        self.power = self.power.transpose(0, 2, 1, 3).reshape(self.power.shape[0], self.power.shape[1], self.power.shape[2]*self.power.shape[3])
        self.phase = self.phase.transpose(0, 2, 1, 3).reshape(self.phase.shape[0], self.phase.shape[1], self.phase.shape[2]*self.phase.shape[3])
        

    def __getitem__(self, idx):
        return (
            self.eeg_data[idx],       # mne.Raw EEG
            self.eye_data[idx],       # mne.Raw EYE
            self.metadata.iloc[idx].to_dict(),
            self.labels[idx],        # pattern_id
            self.imgs[idx],          # torch.Tensor 6x6
            self.power[idx],
            self.phase[idx]
        )

    def __len__(self):
        return len(self.metadata)
    
Dataset = MorletDataset('./Generated/Data_Train/', './Generated/Spectrums/exec_morlets.npz', n_jobs=14, eeg_resampling_freq=256)

In [ ]:
# Dataset = MorletDataset('./Generated/Data_Train/', './Generated/Spectrums/exec_morlets.npz', n_jobs=14, eeg_resampling_freq=256)